### Kinetic and Geometric Feature Extractors

In [ ]:
# Cell 3: Kinetic Feature Extractor
def extract_kinetic_features(motion_sequences):
    """
    Extract kinetic features (velocity, acceleration) from motion sequences
    
    Args:
        motion_sequences: (N, T, J, 3) - N sequences, T frames, J joints, 3D positions
    
    Returns:
        kinetic_features: (N, feature_dim) - flattened kinetic features
    """
    N, T, J, _ = motion_sequences.shape
    
    # Calculate velocities (first derivative)
    velocities = np.diff(motion_sequences, axis=1)  # (N, T-1, J, 3)
    
    # Calculate accelerations (second derivative)
    accelerations = np.diff(velocities, axis=1)  # (N, T-2, J, 3)
    
    # Calculate velocity magnitudes
    vel_magnitudes = np.linalg.norm(velocities, axis=-1)  # (N, T-1, J)
    
    # Calculate acceleration magnitudes
    acc_magnitudes = np.linalg.norm(accelerations, axis=-1)  # (N, T-2, J)
    
    # Aggregate statistics across time
    # Mean and std of velocities and accelerations per joint
    vel_mean = np.mean(vel_magnitudes, axis=1)  # (N, J)
    vel_std = np.std(vel_magnitudes, axis=1)    # (N, J)
    acc_mean = np.mean(acc_magnitudes, axis=1)  # (N, J)
    acc_std = np.std(acc_magnitudes, axis=1)    # (N, J)
    
    # Concatenate all kinetic features
    kinetic_features = np.concatenate([
        vel_mean, vel_std, acc_mean, acc_std
    ], axis=1)  # (N, J*4)
    
    return kinetic_features


In [ ]:
# Cell 4: Geometric Feature Extractor
def extract_geometric_features(motion_sequences):
    """
    Extract geometric features (limb lengths, joint angles, body relationships)
    
    Args:
        motion_sequences: (N, T, J, 3) - N sequences, T frames, J joints, 3D positions
    
    Returns:
        geometric_features: (N, feature_dim) - flattened geometric features
    """
    N, T, J, _ = motion_sequences.shape
    
    
    geometric_features_list = []
    
    for n in range(N):
        sequence = motion_sequences[n]  # (T, J, 3)
        
        # Calculate limb lengths
        limb_lengths = []
        for parent, child in smpl_skeleton:
            limb_vectors = sequence[:, child, :] - sequence[:, parent, :]  # (T, 3)
            lengths = np.linalg.norm(limb_vectors, axis=1)  # (T,)
            limb_lengths.append(np.mean(lengths))  # average over time
        
        # Calculate body extents (bounding box dimensions)
        body_extent_x = np.mean(np.max(sequence[:, :, 0], axis=1) - np.min(sequence[:, :, 0], axis=1))
        body_extent_y = np.mean(np.max(sequence[:, :, 1], axis=1) - np.min(sequence[:, :, 1], axis=1))
        body_extent_z = np.mean(np.max(sequence[:, :, 2], axis=1) - np.min(sequence[:, :, 2], axis=1))
        
        # Calculate center of mass trajectory variance
        com = np.mean(sequence, axis=1)  # (T, 3)
        com_variance = np.var(com, axis=0)  # (3,)
        
        # Concatenate all geometric features for this sequence
        seq_features = np.concatenate([
            limb_lengths,
            [body_extent_x, body_extent_y, body_extent_z],
            com_variance
        ])
        
        geometric_features_list.append(seq_features)
    
    geometric_features = np.array(geometric_features_list)  # (N, feature_dim)
    
    return geometric_features

### Fréchet Inception Distance (FID) Calculation

In [ ]:
# Cell 5: FID Calculation Function
def calculate_frechet_distance(mu1, sigma1, mu2, sigma2, eps=1e-6):
    """
    Calculate Fréchet Distance between two Gaussian distributions
    
    Args:
        mu1: Mean of first distribution (feature_dim,)
        sigma1: Covariance of first distribution (feature_dim, feature_dim)
        mu2: Mean of second distribution (feature_dim,)
        sigma2: Covariance of second distribution (feature_dim, feature_dim)
        eps: Small value for numerical stability
    
    Returns:
        fid_score: Fréchet distance
    """
    # Calculate mean difference
    diff = mu1 - mu2
    
    # Product might be almost singular
    covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)
    
    if not np.isfinite(covmean).all():
        print(f"FID calculation produces NaN. Adding {eps} to diagonal of covariance.")
        offset = np.eye(sigma1.shape[0]) * eps
        covmean = linalg.sqrtm((sigma1 + offset).dot(sigma2 + offset))
    
    # Numerical error might give slight imaginary component
    if np.iscomplexobj(covmean):
        if not np.allclose(np.diagonal(covmean).imag, 0, atol=1e-3):
            m = np.max(np.abs(covmean.imag))
            raise ValueError(f"Imaginary component {m}")
        covmean = covmean.real
    
    tr_covmean = np.trace(covmean)
    
    fid = diff.dot(diff) + np.trace(sigma1) + np.trace(sigma2) - 2 * tr_covmean
    
    return fid


def calculate_fid_score(real_features, generated_features):
    """
    Calculate FID between real and generated motion features
    
    Args:
        real_features: (N_real, feature_dim)
        generated_features: (N_gen, feature_dim)
    
    Returns:
        fid_score: Scalar FID value
    """
    # Calculate statistics for real features
    mu_real = np.mean(real_features, axis=0)
    sigma_real = np.cov(real_features, rowvar=False)
    
    # Calculate statistics for generated features
    mu_gen = np.mean(generated_features, axis=0)
    sigma_gen = np.cov(generated_features, rowvar=False)
    
    # Calculate FID
    fid_score = calculate_frechet_distance(mu_real, sigma_real, mu_gen, sigma_gen)
    
    return fid_score


In [ ]:
# Cell 6: FID Evaluation Wrapper
def evaluate_fid(real_motions, generated_motions):
    """
    Evaluate FID_k and FID_m for motion generation
    
    Args:
        real_motions: (N_real, T, J, 3) - Real motion sequences
        generated_motions: (N_gen, T, J, 3) - Generated motion sequences
    
    Returns:
        results: Dict with FID_k and FID_m scores
    """
    print("Extracting kinetic features...")
    real_kinetic = extract_kinetic_features(real_motions)
    gen_kinetic = extract_kinetic_features(generated_motions)
    
    print("Calculating FID_k...")
    fid_k = calculate_fid_score(real_kinetic, gen_kinetic)
    
    print("Extracting geometric features...")
    real_geometric = extract_geometric_features(real_motions)
    gen_geometric = extract_geometric_features(generated_motions)
    
    print("Calculating FID_m...")
    fid_m = calculate_fid_score(real_geometric, gen_geometric)
    
    results = {
        'FID_k': fid_k,
        'FID_m': fid_m
    }
    
    return results

### Physical Foot Contact Score

In [ ]:
# Cell 7: Foot Contact Detection and Scoring
def detect_foot_contacts(motion_sequences, foot_joint_indices=[7, 8, 10, 11], 
                         velocity_threshold=0.02, height_threshold=0.05):
    """
    Detect foot-ground contacts based on foot velocities and heights
    
    Args:
        motion_sequences: (N, T, J, 3) - Motion sequences
        foot_joint_indices: List of foot joint indices (left_ankle, right_ankle, left_toe, right_toe)
        velocity_threshold: Velocity threshold for contact detection
        height_threshold: Height threshold for ground contact
    
    Returns:
        contact_labels: (N, T, 4) - Binary contact labels for 4 foot points
    """
    N, T, J, _ = motion_sequences.shape
    contact_labels = np.zeros((N, T, len(foot_joint_indices)))
    
    for n in range(N):
        sequence = motion_sequences[n]  # (T, J, 3)
        
        for idx, joint_idx in enumerate(foot_joint_indices):
            joint_positions = sequence[:, joint_idx, :]  # (T, 3)
            
            # Calculate velocities
            velocities = np.diff(joint_positions, axis=0)  # (T-1, 3)
            velocity_magnitudes = np.linalg.norm(velocities, axis=1)  # (T-1,)
            
            # Pad to match original length
            velocity_magnitudes = np.concatenate([[velocity_magnitudes[0]], velocity_magnitudes])
            
            # Get foot heights (Y-axis, assuming Y is up)
            heights = joint_positions[:, 1]
            
            # Contact detected when velocity is low AND height is low
            contacts = (velocity_magnitudes < velocity_threshold) & (heights < height_threshold)
            
            contact_labels[n, :, idx] = contacts.astype(float)
    
    return contact_labels


def calculate_foot_contact_score(motion_sequences, foot_joint_indices=[7, 8, 10, 11]):
    """
    Calculate Physical Foot Contact Score
    
    Args:
        motion_sequences: (N, T, J, 3) - Motion sequences
        foot_joint_indices: List of foot joint indices
    
    Returns:
        contact_score: Average contact consistency score
    """
    contact_labels = detect_foot_contacts(motion_sequences, foot_joint_indices)
    
    # Calculate contact consistency (percentage of frames with reasonable contacts)
    # A "reasonable" contact means at least one foot is on ground when body is low
    
    # Simple version: calculate percentage of time with any foot in contact
    any_contact = np.any(contact_labels, axis=2)  # (N, T)
    contact_rate = np.mean(any_contact)
    
    # More sophisticated: penalize skating (contact label changes while moving fast)
    N, T, num_feet = contact_labels.shape
    skating_penalty = 0
    
    for n in range(N):
        for foot_idx in range(num_feet):
            contacts = contact_labels[n, :, foot_idx]
            
            # Find contact periods
            contact_changes = np.diff(contacts)
            
            # Count number of contact/release transitions (more = more skating)
            num_transitions = np.sum(np.abs(contact_changes))
            skating_penalty += num_transitions
    
    # Normalize skating penalty
    max_possible_transitions = N * T * num_feet
    skating_penalty_normalized = skating_penalty / max_possible_transitions
    
    # Final score: high contact rate, low skating
    foot_contact_score = contact_rate * (1 - skating_penalty_normalized)
    
    return foot_contact_score


### Motion Diversity Metrics

In [ ]:
# Cell 8: Diversity Calculation
def calculate_diversity(features):
    """
    Calculate average pairwise distance (diversity) in feature space
    
    Args:
        features: (N, feature_dim) - Feature vectors
    
    Returns:
        diversity_score: Average Euclidean distance between all pairs
    """
    # Calculate pairwise distances
    distances = pdist(features, metric='euclidean')
    
    # Return mean distance
    diversity_score = np.mean(distances)
    
    return diversity_score


In [ ]:
# Cell 9: Diversity Evaluation Wrapper
def evaluate_diversity(generated_motions):
    """
    Evaluate Div_k and Div_m for generated motions
    
    Args:
        generated_motions: (N, T, J, 3) - Generated motion sequences
    
    Returns:
        results: Dict with Div_k and Div_m scores
    """
    print("Extracting kinetic features for diversity...")
    kinetic_features = extract_kinetic_features(generated_motions)
    div_k = calculate_diversity(kinetic_features)
    
    print("Extracting geometric features for diversity...")
    geometric_features = extract_geometric_features(generated_motions)
    div_m = calculate_diversity(geometric_features)
    
    results = {
        'Div_k': div_k,
        'Div_m': div_m
    }
    
    return results

### Complete Evaluation Pipeline

In [ ]:
# Cell 12: Main Evaluation Function
def evaluate_model(real_motions, generated_motions, lambda_values=None):
    """
    Evaluation pipeline using only 3D motion data (no audio/video required)
    
    Args:
        real_motions: (N_real, T, J, 3) - Real motion sequences from AIST++
        generated_motions: (N_gen, T, J, 3) - Generated motion sequences
        lambda_values: Dict with lambda weights used in training
    
    Returns:
        results: Dict containing evaluation metrics
    """
    print("="*60)
    print("MOTION GENERATION EVALUATION (Motion-Only)")
    print("="*60)
    
    results = {}
    
    # 1. FID Scores (Motion Quality)
    print("\n[1/3] Evaluating Motion Quality (FID)...")
    fid_results = evaluate_fid(real_motions, generated_motions)
    results.update(fid_results)
    print(f"  FID_k (Kinetic): {fid_results['FID_k']:.4f}")
    print(f"  FID_m (Geometric): {fid_results['FID_m']:.4f}")
    
    # 2. Diversity Scores
    print("\n[2/3] Evaluating Motion Diversity...")
    diversity_results = evaluate_diversity(generated_motions)
    results.update(diversity_results)
    print(f"  Div_k (Kinetic): {diversity_results['Div_k']:.4f}")
    print(f"  Div_m (Geometric): {diversity_results['Div_m']:.4f}")
    
    # 3. Foot Contact Score (Physical Plausibility)
    print("\n[3/3] Evaluating Physical Plausibility (Foot Contact)...")
    foot_contact_score = calculate_foot_contact_score(generated_motions)
    results['Foot_Contact'] = foot_contact_score
    print(f"  Foot Contact Score: {foot_contact_score:.4f}")
    
    print("\n" + "="*60)
    print("EVALUATION COMPLETE")
    print("="*60)
    
    # Print summary
    print("\nSUMMARY:")
    for metric, value in results.items():
        print(f"  {metric}: {value:.4f}")
    
    if lambda_values is not None:
        print(f"\nLambda values used: {lambda_values}")
    
    return results

### Visualization of Results

In [ ]:
# Cell 13: Visualize Evaluation Results
def plot_evaluation_results(results, baseline_results=None, save_path='evaluation_results.png'):
    """
    Create visualization of evaluation metrics
    
    Args:
        results: Dict of evaluation results for your model
        baseline_results: Dict of baseline model results (optional)
        save_path: Path to save figure
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    metrics = ['FID_k', 'FID_m', 'Div_k', 'Div_m']
    titles = ['FID Kinetic (↓)', 'FID Geometric (↓)', 
              'Diversity Kinetic (↑)', 'Diversity Geometric (↑)']
    
    for idx, (metric, title) in enumerate(zip(metrics, titles)):
        ax = axes[idx // 2, idx % 2]
        
        values = [results[metric]]
        labels = ['Your Model']
        colors = ['steelblue']
        
        if baseline_results is not None and metric in baseline_results:
            values.append(baseline_results[metric])
            labels.append('Baseline')
            colors.append('coral')
        
        ax.bar(labels, values, color=colors, alpha=0.8)
        ax.set_ylabel('Score', fontsize=12, fontweight='bold')
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.grid(True, alpha=0.3, axis='y')
        
        # Add value labels on bars
        for i, v in enumerate(values):
            ax.text(i, v, f'{v:.4f}', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"Evaluation plot saved to: {save_path}")


### Save and Load Results

In [ ]:
# Cell 16: Save evaluation results
import json

def save_evaluation_results(results, save_path='evaluation_results.json'):
    """Save evaluation results to JSON file"""
    # Convert numpy types to Python types for JSON serialization
    results_serializable = {k: float(v) for k, v in results.items()}
    
    with open(save_path, 'w') as f:
        json.dump(results_serializable, f, indent=4)
    
    print(f"Results saved to: {save_path}")


def load_evaluation_results(load_path='evaluation_results.json'):
    """Load evaluation results from JSON file"""
    with open(load_path, 'r') as f:
        results = json.load(f)
    
    return results


In [ ]:
# Cell 14: Example Usage
"""
# Load your data
real_motions = np.load('aistpp_all_motions.npy')  # (N_real, T, J, 3)
generated_motions = np.load('generated_motions.npy')  # (N_gen, T, J, 3)
audio_paths = ['audio1.wav', 'audio2.wav', ...]  # List of audio files

# Evaluate
lambda_values = {'lambda_joints': 0.01, 'lambda_vel': 1.0, 'lambda_foot': 0.1}
results = evaluate_model(real_motions, generated_motions, audio_paths, lambda_values)

# Visualize
baseline_results = {'FID_k': 30.2, 'FID_m': 11.5, 'Div_k': 8.10, 'Div_m': 4.45}
plot_evaluation_results(results, baseline_results)
"""

## Qualitative Motion Visualizations

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D



def plot_skeleton_3d(joints_3d, title='3D Motion', save_path=None):
    """
    Plot 3D skeleton from joint positions
    joints_3d: (T, J, 3) array where T=frames, J=joints
    """
    fig = plt.figure(figsize=(12, 10))
    
    # Plot multiple frames (e.g., every 10th frame)
    n_frames = min(4, joints_3d.shape[0])
    frame_indices = np.linspace(0, joints_3d.shape[0]-1, n_frames, dtype=int)
    
    for idx, frame_idx in enumerate(frame_indices):
        ax = fig.add_subplot(2, 2, idx+1, projection='3d')
        
        joints = joints_3d[frame_idx]  # (J, 3)
        
        # Plot joints
        ax.scatter(joints[:, 0], joints[:, 1], joints[:, 2], 
                  c='red', s=50, alpha=0.8)
        
        # Plot skeleton connections
        for connection in smpl_skeleton:
            start_joint = joints[connection[0]]
            end_joint = joints[connection[1]]
            ax.plot([start_joint[0], end_joint[0]], 
                   [start_joint[1], end_joint[1]], 
                   [start_joint[2], end_joint[2]], 
                   'b-', linewidth=2, alpha=0.6)
        
        ax.set_xlabel('X')
        ax.set_ylabel('Y')
        ax.set_zlabel('Z')
        ax.set_title(f'Frame {frame_idx}', fontsize=12, fontweight='bold')
        
        # Set equal aspect ratio
        max_range = np.array([joints[:, 0].max()-joints[:, 0].min(),
                             joints[:, 1].max()-joints[:, 1].min(),
                             joints[:, 2].max()-joints[:, 2].min()]).max() / 2.0
        mid_x = (joints[:, 0].max()+joints[:, 0].min()) * 0.5
        mid_y = (joints[:, 1].max()+joints[:, 1].min()) * 0.5
        mid_z = (joints[:, 2].max()+joints[:, 2].min()) * 0.5
        ax.set_xlim(mid_x - max_range, mid_x + max_range)
        ax.set_ylim(mid_y - max_range, mid_y + max_range)
        ax.set_zlim(mid_z - max_range, mid_z + max_range)
    
    plt.suptitle(title, fontsize=16, fontweight='bold')
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

# Example usage 
plot_skeleton_3d(j3d_pos_gt, title='3D Motion Ground Truth')
plot_skeleton_3d(j3d_pos_pe, title='3D Motion Ground Truth')

In [ ]:
def plot_comparison_sequences(gt_motion, gen_motion, frame_indices=[0, 30, 60, 90], 
                              save_path=None):
    """
    Compare ground truth and generated motions side by side
    """
    fig = plt.figure(figsize=(20, 10))
    
    n_frames = len(frame_indices)
    
    for idx, frame_idx in enumerate(frame_indices):
        # Ground truth
        ax1 = fig.add_subplot(2, n_frames, idx+1, projection='3d')
        joints_gt = gt_motion[frame_idx]
        
        ax1.scatter(joints_gt[:, 0], joints_gt[:, 1], joints_gt[:, 2], 
                   c='green', s=50, alpha=0.8, label='GT')
        
        for connection in smpl_skeleton:
            start = joints_gt[connection[0]]
            end = joints_gt[connection[1]]
            ax1.plot([start[0], end[0]], [start[1], end[1]], [start[2], end[2]], 
                    'g-', linewidth=2, alpha=0.6)
        
        ax1.set_title(f'GT - Frame {frame_idx}', fontsize=11, fontweight='bold')
        ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')
        
        # Generated
        ax2 = fig.add_subplot(2, n_frames, n_frames+idx+1, projection='3d')
        joints_gen = gen_motion[frame_idx]
        
        ax2.scatter(joints_gen[:, 0], joints_gen[:, 1], joints_gen[:, 2], 
                   c='red', s=50, alpha=0.8, label='Generated')
        
        for connection in smpl_skeleton:
            start = joints_gen[connection[0]]
            end = joints_gen[connection[1]]
            ax2.plot([start[0], end[0]], [start[1], end[1]], [start[2], end[2]], 
                    'r-', linewidth=2, alpha=0.6)
        
        ax2.set_title(f'Generated - Frame {frame_idx}', fontsize=11, fontweight='bold')
        ax2.set_xlabel('X'); ax2.set_ylabel('Y'); ax2.set_zlabel('Z')
    
    plt.suptitle('Ground Truth vs Generated Motion Comparison', 
                fontsize=16, fontweight='bold')
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

# Example usage 
plot_comparison_sequences(j3d_pos_gt, j3d_pos_pe)

In [ ]:
def plot_trajectory_comparison(gt_motion, gen_motion, joint_idx=0, save_path=None):
    """
    Plot 3D trajectory of specific joint (e.g., root joint)
    """
    fig = plt.figure(figsize=(14, 6))
    
    # 3D trajectory
    ax1 = fig.add_subplot(1, 2, 1, projection='3d')
    
    gt_traj = gt_motion[:, joint_idx, :]
    gen_traj = gen_motion[:, joint_idx, :]
    
    ax1.plot(gt_traj[:, 0], gt_traj[:, 1], gt_traj[:, 2], 
            'g-', linewidth=2, alpha=0.8, label='Ground Truth')
    ax1.plot(gen_traj[:, 0], gen_traj[:, 1], gen_traj[:, 2], 
            'r--', linewidth=2, alpha=0.8, label='Generated')
    
    ax1.scatter(gt_traj[0, 0], gt_traj[0, 1], gt_traj[0, 2], 
               c='green', s=100, marker='o', label='GT Start')
    ax1.scatter(gen_traj[0, 0], gen_traj[0, 1], gen_traj[0, 2], 
               c='red', s=100, marker='o', label='Gen Start')
    
    ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')
    ax1.set_title('3D Root Joint Trajectory', fontsize=13, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2D projection (bird's eye view)
    ax2 = fig.add_subplot(1, 2, 2)
    
    ax2.plot(gt_traj[:, 0], gt_traj[:, 1], 'g-', linewidth=2, alpha=0.8, 
            label='Ground Truth')
    ax2.plot(gen_traj[:, 0], gen_traj[:, 1], 'r--', linewidth=2, alpha=0.8, 
            label='Generated')
    
    ax2.scatter(gt_traj[0, 0], gt_traj[0, 1], c='green', s=100, marker='o')
    ax2.scatter(gen_traj[0, 0], gen_traj[0, 1], c='red', s=100, marker='o')
    
    ax2.set_xlabel('X'); ax2.set_ylabel('Y')
    ax2.set_title('2D Trajectory (Top View)', fontsize=13, fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.axis('equal')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

# Example usage 
plot_trajectory_comparison(j3d_pos_gt, j3d_pos_pe)

In [ ]:
def plot_motion_dynamics(gt_motion, gen_motion, save_path=None):
    """
    Visualize velocity and acceleration profiles
    """
    # Calculate velocities (frame-to-frame differences)
    gt_vel = np.linalg.norm(np.diff(gt_motion, axis=0), axis=-1)  # (T-1, J)
    gen_vel = np.linalg.norm(np.diff(gen_motion, axis=0), axis=-1)
    
    # Calculate accelerations
    gt_acc = np.linalg.norm(np.diff(gt_vel, axis=0), axis=-1)  # (T-2, J)
    gen_acc = np.linalg.norm(np.diff(gen_vel, axis=0), axis=-1)
    
    # Average across joints
    gt_vel_avg = gt_vel.mean(axis=1)
    gen_vel_avg = gen_vel.mean(axis=1)
    gt_acc_avg = gt_acc.mean(axis=1)
    gen_acc_avg = gen_acc.mean(axis=1)
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    
    # Velocity plot
    axes[0].plot(gt_vel_avg, 'g-', linewidth=2, alpha=0.8, label='GT Velocity')
    axes[0].plot(gen_vel_avg, 'r--', linewidth=2, alpha=0.8, label='Gen Velocity')
    axes[0].set_xlabel('Frame', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Average Velocity', fontsize=12, fontweight='bold')
    axes[0].set_title('Motion Velocity Profile', fontsize=14, fontweight='bold')
    axes[0].legend(fontsize=11)
    axes[0].grid(True, alpha=0.3)
    
    # Acceleration plot
    axes[1].plot(gt_acc_avg, 'g-', linewidth=2, alpha=0.8, label='GT Acceleration')
    axes[1].plot(gen_acc_avg, 'r--', linewidth=2, alpha=0.8, label='Gen Acceleration')
    axes[1].set_xlabel('Frame', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('Average Acceleration', fontsize=12, fontweight='bold')
    axes[1].set_title('Motion Acceleration Profile', fontsize=14, fontweight='bold')
    axes[1].legend(fontsize=11)
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

# Example usage 
plot_motion_dynamics(j3d_pos_gt, j3d_pos_pe)

In [ ]:
def plot_foot_contact(motion, foot_contact_labels, save_path=None):
    """
    Visualize foot contact events
    foot_contact_labels: (T, 4) binary labels for [left_foot, left_toe, right_foot, right_toe]
    """
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    
    frames = np.arange(len(foot_contact_labels))
    
    # Left foot contacts
    axes[0].fill_between(frames, 0, foot_contact_labels[:, 0], 
                         alpha=0.6, label='Left Foot', color='blue')
    axes[0].fill_between(frames, 0, foot_contact_labels[:, 1], 
                         alpha=0.4, label='Left Toe', color='lightblue')
    axes[0].set_ylabel('Contact', fontsize=12, fontweight='bold')
    axes[0].set_title('Left Foot Contact Events', fontsize=13, fontweight='bold')
    axes[0].legend(fontsize=10)
    axes[0].grid(True, alpha=0.3)
    axes[0].set_ylim(-0.1, 1.1)
    
    # Right foot contacts
    axes[1].fill_between(frames, 0, foot_contact_labels[:, 2], 
                         alpha=0.6, label='Right Foot', color='red')
    axes[1].fill_between(frames, 0, foot_contact_labels[:, 3], 
                         alpha=0.4, label='Right Toe', color='lightcoral')
    axes[1].set_xlabel('Frame', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('Contact', fontsize=12, fontweight='bold')
    axes[1].set_title('Right Foot Contact Events', fontsize=13, fontweight='bold')
    axes[1].legend(fontsize=10)
    axes[1].grid(True, alpha=0.3)
    axes[1].set_ylim(-0.1, 1.1)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()


## Advanced Visualization with Video Rendering

In [ ]:
# For actual video rendering, consider using:
# - pyrender (for high-quality 3D rendering)
# - trimesh (for mesh manipulation)
# - SMPL-X model for detailed human body

"""
Example workflow for video generation:
1. Load SMPL model parameters
2. For each frame, create SMPL mesh from theta parameters
3. Render mesh from camera viewpoint using pyrender
4. Save frames and compile into video using opencv or moviepy

Note: This requires SMPL model files and proper setup
Refer to: https://github.com/vchoutas/smplx for SMPL implementation
"""

def save_motion_video(motion_sequence, output_path='motion_video.mp4', fps=30):
    """
    Pseudo-code for saving motion as video
    Requires: pyrender, trimesh, smplx, opencv
    """
    import cv2
    # import pyrender
    # import trimesh
    # from smplx import SMPL
    
    # Initialize SMPL model
    # smpl = SMPL(model_path='path/to/smpl', gender='neutral')
    
    frames = []
    
    for frame_idx in range(len(motion_sequence)):
        # Get SMPL parameters for this frame
        # theta = motion_sequence[frame_idx]  # Pose parameters
        
        # Generate mesh
        # output = smpl(body_pose=theta[3:], global_orient=theta[:3])
        # vertices = output.vertices.detach().cpu().numpy()[0]
        # faces = smpl.faces
        
        # Render mesh
        # mesh = trimesh.Trimesh(vertices=vertices, faces=faces)
        # scene = pyrender.Scene()
        # scene.add(pyrender.Mesh.from_trimesh(mesh))
        
        # Add camera and lighting
        # camera = pyrender.PerspectiveCamera(yfov=np.pi / 3.0)
        # light = pyrender.DirectionalLight(color=[1.0, 1.0, 1.0], intensity=3.0)
        
        # Render frame
        # renderer = pyrender.OffscreenRenderer(640, 480)
        # color, _ = renderer.render(scene)
        
        # frames.append(color)
        pass
    
    # Save video using opencv
    # fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    # out = cv2.VideoWriter(output_path, fourcc, fps, (640, 480))
    # for frame in frames:
    #     out.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
    # out.release()
    
    print(f"Video would be saved to: {output_path}")


## Error Analysis Visualizations

In [ ]:
def plot_joint_error_heatmap(gt_motion, gen_motion, joint_names=None, save_path=None):
    """
    Create heatmap showing per-joint errors over time
    """
    # Calculate per-joint per-frame errors
    errors = np.linalg.norm(gt_motion - gen_motion, axis=-1)  # (T, J)
    
    if joint_names is None:
        joint_names = [f'J{i}' for i in range(errors.shape[1])]
    
    fig, ax = plt.subplots(figsize=(16, 8))
    
    im = ax.imshow(errors.T, aspect='auto', cmap='hot', interpolation='nearest')
    
    ax.set_xlabel('Frame', fontsize=12, fontweight='bold')
    ax.set_ylabel('Joint', fontsize=12, fontweight='bold')
    ax.set_title('Per-Joint Position Error Over Time', fontsize=14, fontweight='bold')
    ax.set_yticks(np.arange(len(joint_names)))
    ax.set_yticklabels(joint_names, fontsize=8)
    
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Position Error (units)', fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    # Print statistics
    print(f"\nError Statistics:")
    print(f"Mean error: {errors.mean():.4f}")
    print(f"Std error: {errors.std():.4f}")
    print(f"Max error: {errors.max():.4f}")
    print(f"\nMost problematic joints:")
    joint_errors = errors.mean(axis=0)
    worst_joints = np.argsort(joint_errors)[-5:][::-1]
    for idx in worst_joints:
        print(f"  {joint_names[idx]}: {joint_errors[idx]:.4f}")


In [ ]:
def plot_distribution_comparison(gt_motion, gen_motion, save_path=None):
    """
    Compare statistical distributions of GT vs Generated motions
    """
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    # Joint position distributions
    gt_flat = gt_motion.reshape(-1, 3)
    gen_flat = gen_motion.reshape(-1, 3)
    
    for idx, (ax, coord, label) in enumerate(zip(axes[0], 
                                                  [0, 1, 2], 
                                                  ['X', 'Y', 'Z'])):
        ax.hist(gt_flat[:, coord], bins=50, alpha=0.6, label='GT', 
               color='green', density=True)
        ax.hist(gen_flat[:, coord], bins=50, alpha=0.6, label='Generated', 
               color='red', density=True)
        ax.set_xlabel(f'{label} Coordinate', fontsize=11, fontweight='bold')
        ax.set_ylabel('Density', fontsize=11, fontweight='bold')
        ax.set_title(f'{label}-axis Position Distribution', fontsize=12, fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    # Velocity distributions
    gt_vel = np.linalg.norm(np.diff(gt_motion, axis=0), axis=-1).flatten()
    gen_vel = np.linalg.norm(np.diff(gen_motion, axis=0), axis=-1).flatten()
    
    axes[1, 0].hist(gt_vel, bins=50, alpha=0.6, label='GT', 
                   color='green', density=True)
    axes[1, 0].hist(gen_vel, bins=50, alpha=0.6, label='Generated', 
                   color='red', density=True)
    axes[1, 0].set_xlabel('Velocity Magnitude', fontsize=11, fontweight='bold')
    axes[1, 0].set_ylabel('Density', fontsize=11, fontweight='bold')
    axes[1, 0].set_title('Velocity Distribution', fontsize=12, fontweight='bold')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Acceleration distributions
    gt_acc = np.diff(gt_vel)
    gen_acc = np.diff(gen_vel)
    
    axes[1, 1].hist(gt_acc, bins=50, alpha=0.6, label='GT', 
                   color='green', density=True)
    axes[1, 1].hist(gen_acc, bins=50, alpha=0.6, label='Generated', 
                   color='red', density=True)
    axes[1, 1].set_xlabel('Acceleration', fontsize=11, fontweight='bold')
    axes[1, 1].set_ylabel('Density', fontsize=11, fontweight='bold')
    axes[1, 1].set_title('Acceleration Distribution', fontsize=12, fontweight='bold')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    # Q-Q plot for normality check
    from scipy import stats
    
    stats.probplot(gt_vel, dist="norm", plot=axes[1, 2])
    axes[1, 2].get_lines()[0].set_color('green')
    axes[1, 2].get_lines()[0].set_label('GT')
    
    stats.probplot(gen_vel, dist="norm", plot=axes[1, 2])
    axes[1, 2].get_lines()[2].set_color('red')
    axes[1, 2].get_lines()[2].set_label('Generated')
    
    axes[1, 2].set_title('Q-Q Plot (Velocity)', fontsize=12, fontweight='bold')
    axes[1, 2].legend()
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
